# Course4Ward: fine-tune the Course in the Ward summarizer (Colab)

**Runtime > Change runtime type > T4 GPU** first. Then run the cells top to bottom.

Input: `train.jsonl` and `eval.jsonl` from `fine_tuning_data/` (built by `build_dataset.py`).
Output: a GGUF model + Ollama `Modelfile` (zip download), and `eval_predictions.jsonl` for scoring.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Upload train.jsonl and eval.jsonl from fine_tuning_data/
import re
from google.colab import files
uploaded = files.upload()
# Colab renames re-uploads to "train (2).jsonl"; save them under the names the later cells read.
for name, data in uploaded.items():
    open(re.sub(r" \(\d+\)", "", name), "wb").write(data)
names = {re.sub(r" \(\d+\)", "", n) for n in uploaded}
assert {"train.jsonl", "eval.jsonl"} <= names, "upload both train.jsonl and eval.jsonl"

In [ ]:
from unsloth import FastLanguageModel

# Text-only 4B instruct model with no thinking mode, close to the qwen3.5:4b the service runs.
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 4096  # longest example is about 3.6k tokens

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "eval": "eval.jsonl"})

def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}

ds = ds.map(to_text, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"][-600:])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["eval"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="no",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# Learn only the summary, not the long prompt.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(f"training took {stats.metrics['train_runtime'] / 60:.1f} min")

Eval loss should go down and then level off. If it starts rising while train loss keeps falling, the model is overfitting: rerun with `num_train_epochs=2`.

In [ ]:
# Summarize every held-out group. Score these against the approved summaries for the paper.
import json, time

FastLanguageModel.for_inference(model)
rows = [json.loads(l) for l in open("eval.jsonl", encoding="utf-8")]
with open("eval_predictions.jsonl", "w", encoding="utf-8") as out:
    for row in rows:
        prompt = [m for m in row["messages"] if m["role"] == "user"]
        inputs = tokenizer.apply_chat_template(
            prompt, add_generation_prompt=True, return_tensors="pt",
        ).to("cuda")
        t = time.time()
        gen = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        pred = tokenizer.decode(gen[0][inputs.shape[1]:], skip_special_tokens=True).strip()
        out.write(json.dumps({
            "group_id": row["meta"]["group_id"],
            "reference": row["messages"][-1]["content"],
            "prediction": pred,
            "seconds": round(time.time() - t, 2),
        }, ensure_ascii=False) + "\n")
        print(row["meta"]["group_id"], "\n", pred, "\n")
files.download("eval_predictions.jsonl")

In [ ]:
# Export to GGUF for Ollama (takes ~10-15 min). Unsloth also writes an Ollama Modelfile.
model.save_pretrained_gguf("course4ward", tokenizer, quantization_method="q4_k_m")
!ls -la course4ward*

In [ ]:
!zip -r course4ward-gguf.zip course4ward* -i '*.gguf' '*Modelfile*'
files.download("course4ward-gguf.zip")

## Run it in Course4Ward (on your PC)

1. Unzip, open the folder with the `.gguf` and `Modelfile`, and point the Modelfile's `FROM` line at the `.gguf` file name.
2. `ollama create course4ward -f Modelfile`
3. Start the AI service with `MODEL=course4ward` (PowerShell: `$env:MODEL = "course4ward"`), then `.venv\Scripts\python.exe ollama_client.py`.